# ARCSIX 1-Min Merged Size Distribution Production

This notebook is the single human-facing workflow for the current ARCSIX one-minute merged size-distribution product.

It does the full production sequence in order:

1. Check campaign inputs.
2. Run the one-minute merged size-distribution workflow with temporal regularization.
3. Resume safely from per-chunk NetCDF checkpoints.
4. Run post-merge QC and write QC NetCDF files.
5. Convert the QC NetCDF files to ICARTT R1 files.
6. Make product inspection plots and the campaign-wide reference-instrument comparison figure.

The reusable merge math, instrument readers, QC writers, and ICARTT writer remain in `arcsix_production/arcsix_merge_production.py`. This notebook is the production orchestration record.

## 1. Imports And Paths

Run this cell first. It uses the local `Research` conda environment and imports the ARCSIX production helpers from this repository.

In [1]:
from __future__ import annotations

import csv
import os
import re
import sys
import traceback
from datetime import datetime
from pathlib import Path

REPO = Path('/Users/C832577250/Research/SizeDistMerge')
SRC = REPO / 'src'
for path in (REPO, SRC):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

MPLCONFIGDIR = REPO / '.matplotlib_config'
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from netCDF4 import Dataset

import arcsix_production.arcsix_merge_production as mp
from sizedistmerge.ict_utils import as_timezone_naive_timestamp, drop_timezone_from_index, read_ict_file, read_inlet_flag, read_microphysical
from sizedistmerge.utils import remap_dndlog_by_edges_any

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
})

## 2. Production Settings

These are the current R1 production settings: one-minute windows, temporal regularization on UHSAS RI, POPS RI, and APS density, and a low-size reference selected from FIMS first and PUTLS-MERGE when FIMS is not usable.

In [2]:
DATA_DIR = Path('/Volumes/Hailstone Data/Research Data/ARCSIX_P3B')
PRODUCT_ROOT = Path(
    '/Users/C832577250/Output/'
    'arcsix_sizedist_merge_full_1min_fims_or_putls_temporal_ri0p1_rho5em7'
)

PRODUCT_NAME = 'ARCSIX-MERGED-SIZEDIST-1min'
REVISION = 'R1'

MERGE_SECONDS = 60
FIMS_LAG = 10
INCLOUD_PAD_S = 10
MIN_SAMPLES_PER_INST = 10
MIN_OVERLAP_S = 24
OVERLAP_FREQ = '1s'

PUTLS_REFERENCE_MAX_NM = 300.0
REFERENCE_XMIN = 10.0
REFERENCE_XMAX = 500.0

MOMENT = 'V'
SPACE = 'linear'
PAIR_W = 1.0
BOUNDS_UHSAS = ((1.3, 1.8),)
BOUNDS_APS = ((950.0, 2000.0),)

FINE_BIN = 100
GLOBAL_EDGES = np.logspace(np.log10(10.0), np.log10(5000.0), FINE_BIN + 1).astype(float)
ALPHA_REFERENCE = 1.0
ALPHA_UHSAS = 1.5
ALPHA_POPS = 0.2
ALPHA_APS = 2.0
LAMBDA_TIK = 1e-5
CONSENSUS_C = 0.5
CONSENSUS_DATA_SPACE = 'log10'

TEMPORAL_PRIOR_PARAMS = list(mp.DEFAULT_TEMPORAL_PRIOR_PARAMS)
TEMPORAL_W_UH = 0.1
TEMPORAL_W_PO = 0.1
TEMPORAL_W_RHO = 5e-7
SMOOTH_RHO = True

OPTIMIZER_MAXITER = 100
OPTIMIZER_POPSIZE = 15
OPTIMIZER_POLISH = True
OPTIMIZER_TOL = 1e-6

SAVE_TIME_SERIES_PLOTS = True
SAVE_LOSS_PLOTS = True
SAVE_MERGE_PLOTS = True
RESUME = True
CHECKPOINT_NETCDF = True

EXPECTED_DATES = (
    '20240528', '20240530', '20240531', '20240603', '20240605',
    '20240606', '20240607', '20240610', '20240611', '20240613',
    '20240725', '20240729', '20240730', '20240801', '20240802',
    '20240807', '20240808', '20240809', '20240815',
)

REFERENCE_SOURCES = (
    ('FIMS', 0),
    ('PUTLS_MERGE', 1),
)
NATIVE_INSTRUMENTS = ('UHSAS', 'POPS', 'APS')

PRODUCT_ROOT.mkdir(parents=True, exist_ok=True)
print('Product root:', PRODUCT_ROOT)

Product root: /Users/C832577250/Output/arcsix_sizedist_merge_full_1min_fims_or_putls_temporal_ri0p1_rho5em7


## 3. Input Inventory

This cell checks that each date has PUTLS-MERGE, UHSAS, POPS, and APS inputs. FIMS is not required here because PUTLS-MERGE is the intended reference when FIMS is missing or unusable.

In [ ]:
REQUIRED_INPUTS = {
    'PUTLS_MERGE': 'ARCSIX-PUTLS-MERGE/ARCSIX-PUTLS-MERGE_P3B_{day}_R0.ict',
    'UHSAS': 'PUTLS-UHSAS/ARCSIX-PUTLS-UHSAS_P3B_{day}_R*.ict',
    'POPS': 'PUTLS-POPS/ARCSIX-PUTLS-POPS_P3B_{day}_R*.ict',
    'APS': 'LARGE-APS/ARCSIX-LARGE-APS_P3B_{day}_R*.ict',
}


def discover_available_dates() -> list[str]:
    putls_dir = DATA_DIR / 'ARCSIX-PUTLS-MERGE'
    dates = []
    for path in putls_dir.glob('ARCSIX-PUTLS-MERGE_P3B_2024*_R0.ict'):
        match = re.search(r'_(2024\d{4})_R0\.ict$', path.name)
        if match:
            dates.append(match.group(1))
    return sorted(set(dates))


def missing_required_inputs(day: str) -> dict[str, str]:
    missing = {}
    for name, pattern in REQUIRED_INPUTS.items():
        matches = list(DATA_DIR.glob(pattern.format(day=day)))
        if not matches:
            missing[name] = pattern.format(day=day)
    return missing


def validate_campaign_inputs(dates: list[str]) -> list[str]:
    usable = []
    for day in dates:
        missing = missing_required_inputs(day)
        if missing:
            print(f'{day}: missing {missing}')
            continue
        usable.append(day)
    if not usable:
        raise RuntimeError('No dates with required PUTLS-MERGE/UHSAS/POPS/APS inputs were found.')
    return usable


available_dates = discover_available_dates()
RUN_DATES = validate_campaign_inputs(available_dates)
PRODUCT_ROOT.joinpath('dates.txt').write_text('\n'.join(RUN_DATES) + '\n')
print(f'Usable dates: {len(RUN_DATES)}')
print(RUN_DATES)

## 4. Per-Date Merge Helpers

These helpers load a single day, build one-minute periods, choose FIMS or PUTLS-MERGE as the low-size reference, and preserve per-chunk checkpointing so interrupted runs can resume.

In [ ]:
def date_dash(day: str) -> str:
    return pd.to_datetime(day, format='%Y%m%d').strftime('%Y-%m-%d')


def day_token(date: str) -> str:
    return pd.to_datetime(date).strftime('%Y%m%d')


def load_day_frames(date: str) -> dict[str, pd.DataFrame]:
    frames = mp.read_arcsix_merge_instruments_for_day(
        date,
        DATA_DIR / 'LARGE-APS',
        DATA_DIR / 'FIMS',
        uhsas_dir=DATA_DIR / 'PUTLS-UHSAS',
        pops_dir=DATA_DIR / 'PUTLS-POPS',
        require_fims=False,
    )
    frames = {name: drop_timezone_from_index(df) for name, df in frames.items()}

    if 'FIMS' in frames and frames['FIMS'] is not None and not frames['FIMS'].empty:
        frames['FIMS'] = frames['FIMS'].copy()
        frames['FIMS'].index = frames['FIMS'].index - pd.Timedelta(seconds=FIMS_LAG)

    frames['PUTLS_MERGE'] = mp.read_putls_merge_reference_for_day(
        DATA_DIR,
        date,
        cutoff_nm=PUTLS_REFERENCE_MAX_NM,
    )
    return frames


def build_periods(frames: dict[str, pd.DataFrame], date: str) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    period_frames = {
        name: df
        for name, df in frames.items()
        if name in {'FIMS', 'PUTLS_MERGE', 'UHSAS', 'POPS', 'APS'} and df is not None and not df.empty
    }
    periods = mp.periods_from_frames(period_frames, MERGE_SECONDS)
    if not periods:
        raise RuntimeError(f'No candidate periods were built for {date}.')
    return periods


def completed_periods_and_prior(nc_path: Path) -> tuple[set[int], np.ndarray | None, np.ndarray | None]:
    if not nc_path.exists():
        return set(), None, None
    with Dataset(nc_path) as ds:
        if 'period_idx' not in ds.variables or len(ds.dimensions['chunk']) == 0:
            return set(), None, None
        completed = set(np.asarray(ds.variables['period_idx'][:], dtype=int).tolist())
        common_edges = np.asarray(ds.variables['fine_edges_nm'][:], dtype=float)
        prev = np.asarray([
            float(ds.variables['retrieved_uhsas_n_fit'][-1]),
            float(ds.variables['retrieved_pops_n_fit'][-1]),
            float(ds.variables['retrieved_aps_density'][-1]),
        ])
    return completed, prev, common_edges


def low_data_reason(bin_counts: dict[str, np.ndarray]) -> str | None:
    for name in ('FIMS', 'UHSAS', 'POPS', 'APS'):
        arr = np.asarray(bin_counts.get(name, []), dtype=int)
        if arr.size == 0:
            return f'{name} has no bin-count record'
        nz = arr[arr > 0]
        nz_avg = float(nz.mean()) if nz.size else 0.0
        if nz_avg < MIN_SAMPLES_PER_INST:
            return f'{name} nonzero_avg={nz_avg:.1f} < {MIN_SAMPLES_PER_INST}'
    return None


def candidate_frames(filtered_chunk: dict[str, pd.DataFrame], source_name: str) -> dict[str, pd.DataFrame] | None:
    missing = [name for name in NATIVE_INSTRUMENTS if name not in filtered_chunk or filtered_chunk[name].empty]
    if missing or source_name not in filtered_chunk or filtered_chunk[source_name].empty:
        return None
    return {
        'FIMS': filtered_chunk[source_name],
        'UHSAS': filtered_chunk['UHSAS'],
        'POPS': filtered_chunk['POPS'],
        'APS': filtered_chunk['APS'],
    }


def reference_label(source_flag: int) -> str:
    return 'FIMS' if int(source_flag) == 0 else 'PUTLS-MERGE'


def count_chunks(nc_path: Path) -> int:
    if not nc_path.exists():
        return 0
    with Dataset(nc_path) as ds:
        return len(ds.dimensions['chunk']) if 'chunk' in ds.dimensions else 0

## 5. Run One Date

This is the production merge for one date. It uses the same joint optimization path whether the reference source is FIMS or PUTLS-MERGE. The reference source is recorded in `reference_source_flag` in the NetCDF.

In [ ]:
def run_merge_for_date(date: str) -> Path:
    date = pd.to_datetime(date).strftime('%Y-%m-%d')
    day = day_token(date)
    output_dir = PRODUCT_ROOT
    day_dir = output_dir / date
    totals_dir = day_dir / 'time_series'
    loss_dir = day_dir / 'loss_curve'
    plots_dir = day_dir / 'merge_plots'
    for path in (day_dir, totals_dir, loss_dir, plots_dir):
        path.mkdir(parents=True, exist_ok=True)

    log_file = output_dir / 'output_log.txt'
    error_log = output_dir / 'error_log.txt'
    nc_path = day_dir / f'{date}_sizedist_merged.nc'

    frames = load_day_frames(date)
    periods = build_periods(frames, date)

    inlet_flag = read_inlet_flag(DATA_DIR / 'LARGE-InletFlag', start=date, end=None, prefix='ARCSIX')
    inlet_flag = drop_timezone_from_index(inlet_flag)
    micro = read_microphysical(DATA_DIR / 'LARGE-MICROPHYSICAL', start=date, end=None, prefix='ARCSIX')
    micro = drop_timezone_from_index(micro)
    cpc_total = pd.to_numeric(micro.get('CNgt10nm'), errors='coerce') if not micro.empty else pd.Series(dtype=float)

    completed_period_idx, prev_params, common_edges = completed_periods_and_prior(nc_path) if RESUME else (set(), None, None)
    if prev_params is None:
        prev_params = np.asarray(TEMPORAL_PRIOR_PARAMS, dtype=float)
    if common_edges is None:
        common_edges = GLOBAL_EDGES.copy()

    failed_chunks = []

    with log_file.open('a') as f:
        f.write(f'\nARCSIX {day} 1-min merged size distribution\n')
        f.write(f'Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n')
        f.write(f'DATE: {date}\n')
        f.write(f'OUTPUT_DIR: {output_dir}\n')
        f.write(f'PUTLS_REFERENCE_MAX_NM: {PUTLS_REFERENCE_MAX_NM}\n')
        f.write(f'TEMPORAL_W_UH: {TEMPORAL_W_UH}\n')
        f.write(f'TEMPORAL_W_PO: {TEMPORAL_W_PO}\n')
        f.write(f'TEMPORAL_W_RHO: {TEMPORAL_W_RHO}\n')
        f.write(f'RESUME_COMPLETED_PERIODS: {len(completed_period_idx)}\n\n')

    for i, (t_start, t_end) in enumerate(periods):
        if RESUME and i in completed_period_idx:
            continue

        try:
            raw_chunk = {
                name: df.loc[t_start:t_end]
                for name, df in frames.items()
                if df is not None and not df.empty and not df.loc[t_start:t_end].empty
            }
            if not raw_chunk:
                with log_file.open('a') as f:
                    f.write(f'[{i:03d}] SKIP empty period {t_start} -> {t_end}\n')
                continue

            inc_flag = mp.chunk_is_incloud(inlet_flag, t_start, t_end, tol_s=INCLOUD_PAD_S)
            inlet_chunk = inlet_flag.loc[t_start:t_end] if not inlet_flag.empty else inlet_flag.iloc[0:0]
            cpc_chunk = cpc_total.loc[t_start:t_end] if not cpc_total.empty else cpc_total.iloc[0:0]

            filtered_chunk = mp.filter_chunk_by_inlet_flag(
                raw_chunk,
                inlet_flag,
                t_start,
                t_end,
                pad_s=INCLOUD_PAD_S,
            )
            filtered_chunk = {name: df for name, df in filtered_chunk.items() if df is not None and not df.empty}
            if not filtered_chunk:
                with log_file.open('a') as f:
                    f.write(f'[{i:03d}] SKIP all data removed by inlet flag\n')
                continue

            accepted_this_period = False
            for source_name, source_flag in REFERENCE_SOURCES:
                required_for_overlap = (source_name, 'UHSAS', 'POPS', 'APS')
                candidate = candidate_frames(filtered_chunk, source_name)
                if candidate is None:
                    continue

                overlap_s = mp.overlap_seconds_all_instruments(
                    filtered_chunk,
                    t_start,
                    t_end,
                    instruments=required_for_overlap,
                    freq=OVERLAP_FREQ,
                )
                if overlap_s < MIN_OVERLAP_S:
                    with log_file.open('a') as f:
                        f.write(
                            f'[{i:03d}] {reference_label(source_flag)} rejected: '
                            f'overlap {overlap_s}s < {MIN_OVERLAP_S}s\n'
                        )
                    continue

                specs, line_kwargs, fill_kwargs, bin_counts = mp.make_filtered_specs(candidate, log_file)
                if any(name not in specs for name in ('FIMS', 'UHSAS', 'POPS', 'APS')):
                    continue
                reason = low_data_reason(bin_counts)
                if reason is not None:
                    with log_file.open('a') as f:
                        f.write(f'[{i:03d}] {reference_label(source_flag)} rejected: {reason}\n')
                    continue

                if SAVE_TIME_SERIES_PLOTS:
                    fig_ts, _ = mp.plot_period_totals(
                        dict(candidate),
                        title=f'{date} period {i:03d} reference={reference_label(source_flag)}',
                        inlet_flag=inlet_flag,
                        gauss_win=10,
                        gauss_std=2,
                        cpc_total=cpc_chunk,
                        t_start=t_start,
                        t_end=t_end,
                    )
                    fig_ts.savefig(totals_dir / f'sizedist_{i:03d}_totals.png', dpi=150)
                    plt.close(fig_ts)

                specs_opt, line_kwargs_opt, fill_kwargs_opt, opt_res = mp.run_joint_optimization(
                    specs,
                    line_kwargs,
                    fill_kwargs,
                    moment=MOMENT,
                    space=SPACE,
                    pair_w=PAIR_W,
                    uhsas_bounds=list(BOUNDS_UHSAS),
                    aps_bounds=list(BOUNDS_APS),
                    uhsas_xmin=None,
                    uhsas_xmax=None,
                    fims_xmin=REFERENCE_XMIN,
                    fims_xmax=REFERENCE_XMAX,
                    pops_xmin=None,
                    pops_xmax=None,
                    lut_dir=None,
                    pops_ri_src=None,
                    w_uhsas=1.0,
                    w_pops=1.0,
                    w_aps=1.0,
                    temporal_w_uh=TEMPORAL_W_UH,
                    temporal_w_po=TEMPORAL_W_PO,
                    temporal_w_rho=TEMPORAL_W_RHO,
                    prev_params=prev_params,
                    smooth_rho=SMOOTH_RHO,
                    optimizer_maxiter=OPTIMIZER_MAXITER,
                    optimizer_tol=OPTIMIZER_TOL,
                    optimizer_popsize=OPTIMIZER_POPSIZE,
                    optimizer_polish=OPTIMIZER_POLISH,
                )
                prev_params = np.asarray([opt_res['n_fit'], opt_res['n_pops_fit'], opt_res['rho_fit']], dtype=float)

                if SAVE_LOSS_PLOTS:
                    fig_loss, ax_loss = mp.plot_history(opt_res['hist'])
                    ax_loss.set_title(f'opt hist {date} {i:03d} {reference_label(source_flag)}')
                    fig_loss.savefig(loss_dir / f'sizedist_{i:03d}_opt_hist.png', dpi=150)
                    plt.close(fig_loss)

                uh_label = opt_res['fit_labels'].get('UHSAS')
                po_label = opt_res['fit_labels'].get('POPS')
                aps_label = opt_res['fit_labels']['APS']

                tik_specs, tik_lines, tik_fills, _tik_diag = mp.make_consensus_merged_spec(
                    e_fims_sel=specs_opt['FIMS_applied'][1],
                    y_fims_sel=specs_opt['FIMS_applied'][2],
                    e_uhsas_fit=specs_opt[uh_label][1],
                    y_uhsas_fit=specs_opt[uh_label][2],
                    e_pops_fit=specs_opt[po_label][1],
                    y_pops_fit=specs_opt[po_label][2],
                    e_aps_fit=specs_opt[aps_label][1],
                    y_aps_fit=specs_opt[aps_label][2],
                    lam=LAMBDA_TIK,
                    n_points=FINE_BIN,
                    alpha_fims=ALPHA_REFERENCE,
                    alpha_uhsas=ALPHA_UHSAS,
                    alpha_pops=ALPHA_POPS,
                    alpha_aps=ALPHA_APS,
                    c_punish=CONSENSUS_C,
                    data_space=CONSENSUS_DATA_SPACE,
                )
                specs_opt.update(tik_specs)
                line_kwargs_opt.update(tik_lines)
                fill_kwargs_opt.update(tik_fills)

                if SAVE_MERGE_PLOTS:
                    (fig_n, _), (fig_v, _), _ = mp.plot_sizedist_all(
                        specs=specs_opt,
                        merged_spec=tik_specs,
                        line_kwargs=line_kwargs_opt,
                        merged_line_kwargs=tik_lines,
                        fill_kwargs=fill_kwargs_opt,
                        merged_fill_kwargs=tik_fills,
                        inlet_flag=inlet_chunk,
                        d_str=f'{date} {reference_label(source_flag)}',
                    )
                    fig_n.savefig(plots_dir / f'{date}_chunk{i:03d}_dNdlogDp.png', dpi=200)
                    fig_v.savefig(plots_dir / f'{date}_chunk{i:03d}_dVdlogDp.png', dpi=200)
                    plt.close(fig_n)
                    plt.close(fig_v)

                tik_name = next(iter(tik_specs.keys()))
                tik_edges = tik_specs[tik_name][1]
                tik_vals = tik_specs[tik_name][2]

                ref_edges = specs_opt['FIMS_applied'][1]
                ref_vals = specs_opt['FIMS_applied'][2]
                uh_edges = specs_opt[uh_label][1]
                uh_vals = specs_opt[uh_label][2]
                po_edges = specs_opt[po_label][1]
                po_vals = specs_opt[po_label][2]
                aps_edges = specs_opt[aps_label][1]
                aps_vals = specs_opt[aps_label][2]

                ref_on_common = remap_dndlog_by_edges_any(ref_edges, common_edges, ref_vals)
                uhsas_on_common = remap_dndlog_by_edges_any(uh_edges, common_edges, uh_vals)
                pops_on_common = remap_dndlog_by_edges_any(po_edges, common_edges, po_vals)
                aps_on_common = remap_dndlog_by_edges_any(aps_edges, common_edges, aps_vals)
                merged_on_common = remap_dndlog_by_edges_any(tik_edges, common_edges, tik_vals)

                if CHECKPOINT_NETCDF:
                    mp.append_day_netcdf_checkpoint(
                        day_dir,
                        date,
                        fine_edges=common_edges,
                        fims_aligned=ref_on_common,
                        uhsas_aligned=uhsas_on_common,
                        pops_aligned=pops_on_common,
                        aps_aligned=aps_on_common,
                        merged=merged_on_common,
                        t_start=t_start,
                        t_end=t_end,
                        incloud_flag=inc_flag,
                        n_fit=opt_res['n_fit'],
                        n_pops_fit=opt_res['n_pops_fit'],
                        rho_fit=opt_res['rho_fit'],
                        best_cost=opt_res['best_cost'],
                        period_idx=i,
                        chunk_number=i,
                        reference_source_flag=source_flag,
                        orig_APS_edges=aps_edges,
                        orig_UHSAS_edges=uh_edges,
                        orig_POPS_edges=po_edges,
                        orig_FIMS_edges=ref_edges,
                    )

                accepted_this_period = True
                with log_file.open('a') as f:
                    f.write(
                        f'[{i:03d}] ACCEPT reference={reference_label(source_flag)} '
                        f'n_uh={opt_res["n_fit"]:.4f} n_po={opt_res["n_pops_fit"]:.4f} '
                        f'rho={opt_res["rho_fit"]:.1f} cost={opt_res["best_cost"]:.6g}\n'
                    )
                break

            if not accepted_this_period:
                with log_file.open('a') as f:
                    f.write(f'[{i:03d}] SKIP no usable FIMS or PUTLS-MERGE reference with UHSAS/POPS/APS\n')

        except Exception as exc:
            failed_chunks.append((i, type(exc).__name__, str(exc)))
            with error_log.open('a') as ef:
                ef.write(f'[{datetime.now():%Y-%m-%d %H:%M:%S}] ERROR period {i:03d} {t_start} -> {t_end}\n')
                ef.write(f'{type(exc).__name__}: {exc}\n')
                ef.write(traceback.format_exc())
                ef.write('\n')

    if failed_chunks:
        preview = '; '.join(f'{i:03d} {typ}: {msg}' for i, typ, msg in failed_chunks[:3])
        raise RuntimeError(f'{len(failed_chunks)} merge period(s) failed. First failures: {preview}. See {error_log}.')

    print(f'{date}: accepted rows now in NetCDF: {count_chunks(nc_path)}')
    print('NetCDF:', nc_path)
    return nc_path

## 6. Run Merge Batch From This Notebook

Set `DATES_TO_RUN` before running this cell. Use `RUN_DATES` for the full campaign or a short list such as `['20240605']` for a test. The batch run is intentionally a notebook cell, not a separate driver script.

In [ ]:
DATES_TO_RUN = RUN_DATES  # full campaign; replace with ['20240605'] for a single-date test

merge_outputs = []
for day in DATES_TO_RUN:
    merge_outputs.append(run_merge_for_date(date_dash(day)))

print(f'Finished {len(merge_outputs)} date(s).')

## 7. Post-Merge QC And ICARTT Conversion

This cell runs the same product-stage QC and ICARTT conversion used for the R1 archive. It consumes the daily merge NetCDF files under `PRODUCT_ROOT`, writes QC NetCDF copies, and then writes R1 ICARTT files.

In [ ]:
DATA_SOURCE_DESCRIPTION = (
    'One-minute merged aerosol size distribution from FIMS or PUTLS-MERGE '
    'reference spectra with UHSAS, POPS, and APS, NASA P-3B aircraft'
)

REVISION_COMMENT = (
    'Revision 1. One-minute merged aerosol number size distribution product '
    'from FIMS, PUTLS-MERGE, UHSAS, POPS, and APS. FIMS is used as the reference '
    'spectrum where available; PUTLS-MERGE is used as the reference when FIMS is '
    'not usable. UHSAS, POPS, and APS are aligned through the joint optimization '
    'path with temporal regularization weights temporal_w_uh=0.1, '
    'temporal_w_po=0.1, and temporal_w_rho=5e-7. The reference_source_flag column '
    'identifies the reference used for each row: 0=FIMS, 1=PUTLS-MERGE. Post-merge '
    'QC follows the previous ARCSIX workflow: warning_high_cost is set when '
    'optimization_best_cost > 0.2; warning_merged_gt10_diff_from_cpc is based on '
    'the robust linear residual merged_total_gt10nm - CPC_median with K_WARN=10; '
    'extreme residual rows are removed from the QC NetCDF and ICARTT outputs with '
    'K_DROP=20. Retrieved RI and APS-density columns are experimental/diagnostic only. '
    'For analyses of Dp < 50 nm, use the native FIMS or PUTLS-MERGE data where '
    'appropriate.'
)


def _date_from_netcdf(path: Path) -> str:
    return path.name.split('_', 1)[0].replace('-', '')


def validate_merge_netcdf_set(base_dir: Path, expected_dates: tuple[str, ...] = EXPECTED_DATES) -> tuple[Path, ...]:
    nc_files = tuple(mp.find_merged_netcdf_files(base_dir))
    dates = tuple(_date_from_netcdf(path) for path in nc_files)
    missing = sorted(set(expected_dates) - set(dates))
    extra = sorted(set(dates) - set(expected_dates))
    duplicates = sorted(date for date in set(dates) if dates.count(date) > 1)
    if missing or extra or duplicates:
        raise RuntimeError(
            'Unexpected merge NetCDF set. '
            f'missing={missing}, extra={extra}, duplicates={duplicates}, '
            f'found={len(nc_files)}, expected={len(expected_dates)}'
        )
    return nc_files


def run_qc_and_icartt() -> tuple[mp.PostMergeQCResult, tuple[Path, ...]]:
    nc_files = validate_merge_netcdf_set(PRODUCT_ROOT)
    print(f'QC input root: {PRODUCT_ROOT}')
    print(f'Found {len(nc_files)} daily merge NetCDF files.')

    qc = mp.run_post_merge_product_qc(
        base_dir=PRODUCT_ROOT,
        data_dir=DATA_DIR,
        high_cost_thresh=0.2,
        cutoff_nm=10.0,
        k_sigma_warn=10.0,
        k_sigma_drop=20.0,
        min_points_for_robust=50,
        cpc_column='CNgt10nm',
        cpc_statistic='median',
        plot_lo=0.0,
        plot_hi=8000.0,
    )

    print(f'QC table: {qc.qc_table_path}')
    print(f'QC plots: {qc.qc_plot_dir}')
    print(f'QC NetCDF: {qc.qc_netcdf_dir}')
    print(
        'QC chunks: '
        f'total={qc.total_chunks}, kept={qc.kept_chunks}, '
        f'dropped_extreme={qc.dropped_extreme_chunks}'
    )

    ict_paths = mp.convert_qc_netcdf_to_icartt(
        input_dir=qc.qc_netcdf_dir,
        out_dir=PRODUCT_ROOT / 'icartt_from_qc_flagged_nc',
        product_name=PRODUCT_NAME,
        revision=REVISION,
        data_source_desc=DATA_SOURCE_DESCRIPTION,
        revision_comment=REVISION_COMMENT,
    )

    print(f'ICARTT output: {PRODUCT_ROOT / "icartt_from_qc_flagged_nc"}')
    print(f'Wrote {len(ict_paths)} {REVISION} ICARTT files.')
    return qc, ict_paths


qc_result, ict_paths = run_qc_and_icartt()

## 8. Product-Level ICARTT Inspection Plots

These plots check the delivered ICT files: daily spectra, daily time-diameter heatmaps, all-day geometric means, bin-edge consistency, variable descriptions, and QC warning counts.

In [ ]:
ICT_DIR = PRODUCT_ROOT / 'icartt_from_qc_flagged_nc'
ICT_PRODUCT_DIR = ICT_DIR / f'{PRODUCT_NAME}_P3B'
QC_TABLE = PRODUCT_ROOT / 'qc_plots' / 'per_chunk_qc_with_flags.csv'
INSPECTION_PLOT_DIR = ICT_DIR / '_plots_edges_check'
FILL_THRESH = -9000.0


def read_icartt_product(path: Path) -> tuple[list[str], list[str], np.ndarray]:
    lines = path.read_text(errors='replace').splitlines()
    if not lines:
        raise RuntimeError(f'Empty ICARTT file: {path}')
    nheader = int(lines[0].split(',', 1)[0].strip())
    if nheader <= 0 or nheader > len(lines):
        raise RuntimeError(f'{path.name}: invalid header line count {nheader}')
    header = lines[:nheader]
    columns = [part.strip() for part in header[-1].split(',')]
    data = np.genfromtxt(path, delimiter=',', skip_header=nheader)
    if data.ndim == 1:
        data = data[None, :]
    if data.shape[1] != len(columns):
        raise RuntimeError(f'{path.name}: data has {data.shape[1]} columns but header lists {len(columns)}')
    return header, columns, data


def parse_header_float_list(header: list[str], tag: str) -> np.ndarray | None:
    for line in header:
        if tag not in line:
            continue
        text = line.split(tag, 1)[1].strip()
        if text.startswith(':'):
            text = text[1:].strip()
        values = [part.strip() for part in text.split(',') if part.strip()]
        return np.array([float(value) for value in values], dtype=float)
    return None


def dnlog_indices(columns: list[str]) -> list[int]:
    def key(index: int) -> int:
        match = re.match(r'DNLOG_(\d+)', columns[index])
        return int(match.group(1)) if match else 10**9
    indices = [idx for idx, name in enumerate(columns) if name.startswith('DNLOG_')]
    if not indices:
        raise RuntimeError('No DNLOG_### columns found in ICARTT header')
    return sorted(indices, key=key)


def centers_from_header(header: list[str], n_bins: int) -> tuple[np.ndarray, np.ndarray]:
    edges = parse_header_float_list(header, 'FINE_EDGES_NM')
    centers = parse_header_float_list(header, 'FINE_CENTERS_NM')
    if edges is not None:
        if edges.size != n_bins + 1:
            raise RuntimeError(f'FINE_EDGES_NM has {edges.size} values for {n_bins} bins')
        if np.any(np.diff(edges) <= 0):
            raise RuntimeError('FINE_EDGES_NM is not strictly increasing')
        return np.sqrt(edges[:-1] * edges[1:]), edges
    if centers is None:
        raise RuntimeError('Header has neither FINE_EDGES_NM nor FINE_CENTERS_NM')
    if centers.size != n_bins:
        raise RuntimeError(f'FINE_CENTERS_NM has {centers.size} values for {n_bins} bins')
    if np.any(np.diff(centers) <= 0):
        raise RuntimeError('FINE_CENTERS_NM is not strictly increasing')
    edges = np.empty(centers.size + 1, dtype=float)
    internal_edges = np.sqrt(centers[:-1] * centers[1:])
    edges[1:-1] = internal_edges
    edges[0] = centers[0] ** 2 / internal_edges[0]
    edges[-1] = centers[-1] ** 2 / internal_edges[-1]
    return centers, edges


def nan_geomean(values: np.ndarray) -> np.ndarray:
    valid = np.isfinite(values) & (values > 0.0)
    out = np.full(values.shape[1], np.nan, dtype=float)
    for idx in range(values.shape[1]):
        col = values[valid[:, idx], idx]
        if col.size:
            out[idx] = float(np.exp(np.mean(np.log(col))))
    return out


def load_ict_spectrum(path: Path) -> dict[str, object]:
    header, columns, data = read_icartt_product(path)
    dn_idx = dnlog_indices(columns)
    centers, edges = centers_from_header(header, len(dn_idx))
    spectra = data[:, dn_idx].astype(float)
    spectra[spectra <= FILL_THRESH] = np.nan
    if not np.any(np.isfinite(spectra)):
        raise RuntimeError(f'{path.name}: all DNLOG values are fill or NaN')
    time_start = data[:, columns.index('Time_Start')].astype(float)
    time_start[time_start <= FILL_THRESH] = np.nan
    return {
        'path': path,
        'header': header,
        'columns': columns,
        'data': data,
        'centers': centers,
        'edges': edges,
        'spectra': spectra,
        'time_start': time_start,
        'geomean': nan_geomean(spectra),
    }


def plot_daily_spectrum(product: dict[str, object], out_dir: Path) -> Path:
    path = product['path']
    centers = np.asarray(product['centers'], dtype=float)
    spectra = np.asarray(product['spectra'], dtype=float)
    geomean = np.asarray(product['geomean'], dtype=float)
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    for row in spectra:
        ax.plot(centers, row, alpha=0.4, linewidth=1.0)
    ax.plot(centers, geomean, color='black', linewidth=2.5, label='Geometric mean')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Dp (nm) [bin centers]')
    ax.set_ylabel('dN/dlog10(Dp) (#/cm^3)')
    ax.set_title(path.name)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path = out_dir / f'{path.stem}_all_spectra_geomean.png'
    fig.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return out_path


def plot_ict_heatmap(product: dict[str, object], out_dir: Path) -> Path:
    path = product['path']
    date_token = re.search(r'(\d{8})', path.name)
    if not date_token:
        raise RuntimeError(f'Could not parse date from {path.name}')
    day = pd.to_datetime(date_token.group(1), format='%Y%m%d')
    seconds = np.asarray(product['time_start'], dtype=float)
    centers = np.asarray(product['centers'], dtype=float)
    spectra = np.asarray(product['spectra'], dtype=float)
    with np.errstate(invalid='ignore'):
        z = np.log10(spectra.T)
    times = day + pd.to_timedelta(seconds, unit='s')
    fig, ax = plt.subplots(figsize=(8.0, 4.8))
    mesh = ax.pcolormesh(times, centers, z, shading='nearest', cmap='viridis', vmin=-4, vmax=4)
    ax.set_yscale('log')
    ax.set_ylim(10, 5000)
    ax.set_ylabel('Dp (nm)')
    ax.set_xlabel('Time (UTC)')
    ax.set_title(f'{date_token.group(1)} ARCSIX 1-min R1 merged dN/dlog10(Dp)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    fig.colorbar(mesh, ax=ax, label='log10(dN/dlog10(Dp))')
    fig.tight_layout()
    out_path = out_dir / f'{path.stem}_time_diameter_heatmap.png'
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    return out_path


def plot_all_daily_geomeans(products: list[dict[str, object]], out_dir: Path) -> Path:
    fig, ax = plt.subplots(figsize=(7.2, 5.2))
    cmap = plt.get_cmap('tab20')
    for idx, product in enumerate(products):
        path = product['path']
        date_match = re.search(r'(\d{8})', path.name)
        label = date_match.group(1) if date_match else path.stem
        ax.plot(
            np.asarray(product['centers'], dtype=float),
            np.asarray(product['geomean'], dtype=float),
            linewidth=1.5,
            color=cmap(idx % cmap.N),
            label=label,
        )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(10, 5000)
    ax.set_ylim(1e-4, 1e5)
    ax.set_xlabel('Dp (nm)')
    ax.set_ylabel('Daily geometric mean dN/dlog10(Dp) (# cm$^{-3}$)')
    ax.set_title('ARCSIX 1-min R1 merged size distribution: daily geometric means')
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    out_path = out_dir / 'all_days_daily_geomean_spectra.png'
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    return out_path


def write_edges_report(products: list[dict[str, object]], out_dir: Path) -> Path:
    reference = products[0]
    ref_path = reference['path']
    ref_edges = np.asarray(reference['edges'], dtype=float)
    rows = []
    all_same = True
    for product in products:
        path = product['path']
        edges = np.asarray(product['edges'], dtype=float)
        if edges.shape != ref_edges.shape:
            all_same = False
            rows.append((path.name, 'shape_mismatch', ''))
            continue
        max_abs_diff = float(np.max(np.abs(edges - ref_edges)))
        if max_abs_diff != 0.0:
            all_same = False
        rows.append((path.name, 'max_abs_diff_nm', f'{max_abs_diff:.12g}'))
    out_path = out_dir / 'edges_check_report.txt'
    with out_path.open('w') as handle:
        handle.write(f'ICT_DIR: {ICT_DIR}\n')
        handle.write(f'Reference: {ref_path.name}\n')
        handle.write(f'All edges identical across files? {all_same}\n\n')
        for file_name, kind, value in rows:
            handle.write(f'{file_name}: {kind} = {value}\n')
    return out_path


def write_variable_description_csv(ict_path: Path, out_dir: Path) -> Path:
    header, _columns, _data = read_icartt_product(ict_path)
    n_dep = int(header[9].strip())
    definition_lines = [header[8], *header[12 : 12 + n_dep]]
    out_path = out_dir / f'{ict_path.stem}_variable_descriptions.csv'
    with out_path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['Variable', 'Units', 'Standard_Variable_Name', 'Description'])
        writer.writeheader()
        for line in definition_lines:
            parts = [part.strip() for part in line.split(',', 3)]
            if len(parts) != 4:
                raise RuntimeError(f'Bad variable definition line: {line!r}')
            writer.writerow(dict(zip(['Variable', 'Units', 'Standard_Variable_Name', 'Description'], parts)))
    return out_path


def plot_qc_warning_counts(out_dir: Path) -> Path:
    table = pd.read_csv(QC_TABLE)
    grouped = table.groupby('date', sort=True).agg(
        total=('date', 'size'),
        warning_high_cost=('warning_high_cost', 'sum'),
        warning_merged_gt10_diff_from_cpc=('warning_merged_gt10_diff_from_cpc', 'sum'),
    )
    grouped['any_warning'] = (
        table.assign(
            any_warning=lambda df: (
                df['warning_high_cost'].astype(bool)
                | df['warning_merged_gt10_diff_from_cpc'].astype(bool)
            ).astype(int)
        )
        .groupby('date')['any_warning']
        .sum()
    )
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    x = np.arange(grouped.shape[0])
    ax.bar(x, grouped['total'], color='0.82', label='QC table rows')
    ax.bar(x, grouped['any_warning'], color='tab:orange', label='any warning')
    ax.bar(x, grouped['warning_merged_gt10_diff_from_cpc'], color='tab:red', label='merged-vs-CPC warning')
    ax.set_xticks(x)
    ax.set_xticklabels([date.replace('-', '') for date in grouped.index], rotation=45, ha='right')
    ax.set_ylabel('Rows')
    ax.set_title('ARCSIX 1-min R1 QC warning counts before extreme-row removal')
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0)
    fig.tight_layout(rect=(0.0, 0.0, 0.82, 1.0))
    out_path = out_dir / 'qc_warning_counts_by_date.png'
    fig.savefig(out_path, dpi=200)
    plt.close(fig)
    grouped.to_csv(out_dir / 'qc_warning_counts_by_date.csv')
    return out_path


def make_ict_inspection_plots() -> dict[str, object]:
    ict_files = sorted(ICT_PRODUCT_DIR.glob('*.ict'))
    if not ict_files:
        raise RuntimeError(f'No ICARTT files found in {ICT_PRODUCT_DIR}')
    INSPECTION_PLOT_DIR.mkdir(parents=True, exist_ok=True)
    products = [load_ict_spectrum(path) for path in ict_files]
    daily_plot_paths = [plot_daily_spectrum(product, INSPECTION_PLOT_DIR) for product in products]
    heatmap_paths = [plot_ict_heatmap(product, INSPECTION_PLOT_DIR) for product in products]
    all_days_plot = plot_all_daily_geomeans(products, INSPECTION_PLOT_DIR)
    edges_report = write_edges_report(products, INSPECTION_PLOT_DIR)
    variable_csv = write_variable_description_csv(ict_files[0], INSPECTION_PLOT_DIR)
    warning_plot = plot_qc_warning_counts(INSPECTION_PLOT_DIR)
    print(f'ICARTT files: {len(ict_files)}')
    print(f'Daily spectra plots: {len(daily_plot_paths)}')
    print(f'Time-diameter heatmaps: {len(heatmap_paths)}')
    print(f'All-days geometric mean plot: {all_days_plot}')
    print(f'QC warning-count plot: {warning_plot}')
    print(f'Variable-description CSV: {variable_csv}')
    print(f'Edges report: {edges_report}')
    return {
        'daily_plot_paths': daily_plot_paths,
        'heatmap_paths': heatmap_paths,
        'all_days_plot': all_days_plot,
        'edges_report': edges_report,
        'variable_csv': variable_csv,
        'warning_plot': warning_plot,
    }


ict_inspection = make_ict_inspection_plots()

## 9. Retrieved-Parameter Time Series Diagnostics

This reproduces the previous-style RI, APS-density, and ambient-RH diagnostic plot for one or more dates. PUTLS-MERGE reference periods are shaded blue. DASH is plotted only when that date has a DASH file.

In [ ]:
MA_WINDOW = '10min'
MAX_LINE_GAP = pd.Timedelta('2min')
RI_AVG_MIN = 1.32
RI_AVG_MAX = 1.78


def base_time(ds: Dataset) -> pd.Timestamp:
    return as_timezone_naive_timestamp(getattr(ds, 'base_time_iso'))


def merge_product_frame(nc_path: Path) -> pd.DataFrame:
    with Dataset(nc_path) as ds:
        base = base_time(ds)
        start_s = np.asarray(ds.variables['time_start_since_base_s'][:], float)
        end_s = np.asarray(ds.variables['time_end_since_base_s'][:], float)
        times = base + pd.to_timedelta(0.5 * (start_s + end_s), unit='s')
        out = pd.DataFrame(
            {
                'uhsas_n': np.asarray(ds.variables['retrieved_uhsas_n_fit'][:], float),
                'pops_n': np.asarray(ds.variables['retrieved_pops_n_fit'][:], float),
                'aps_density': np.asarray(ds.variables['retrieved_aps_density'][:], float),
                'reference_source_flag': np.asarray(ds.variables['reference_source_flag'][:], int),
                't_start': base + pd.to_timedelta(start_s, unit='s'),
                't_end': base + pd.to_timedelta(end_s, unit='s'),
            },
            index=pd.DatetimeIndex(times, name='time'),
        )
    return out.replace([np.inf, -np.inf], np.nan).sort_index()


def split_by_time_gaps(series: pd.Series) -> list[pd.Series]:
    series = series.dropna().sort_index()
    if len(series) < 2:
        return [series]
    breaks = np.flatnonzero(series.index.to_series().diff().to_numpy() > MAX_LINE_GAP)
    return [series.iloc[a:b] for a, b in zip(np.r_[0, breaks], np.r_[breaks, len(series)])]


def plot_segmented_line(ax, series: pd.Series, *, color, lw, alpha, label=None) -> None:
    first = True
    for part in split_by_time_gaps(series):
        if len(part) >= 2:
            ax.plot(part.index, part.values, color=color, lw=lw, alpha=alpha, label=label if first else '_nolegend_')
            first = False


def moving_average(series: pd.Series) -> pd.Series:
    return series.dropna().sort_index().rolling(MA_WINDOW, min_periods=2).mean().dropna()


def ri_for_average(series: pd.Series) -> pd.Series:
    series = series.dropna().sort_index()
    return series[(series >= RI_AVG_MIN) & (series <= RI_AVG_MAX)]


def dash_groups_by_dp(dash_df: pd.DataFrame) -> list[tuple[float, pd.Series]]:
    tmp = dash_df.copy()
    tmp['_Dp'] = pd.to_numeric(tmp['Dp'], errors='coerce')
    tmp['_RI'] = pd.to_numeric(tmp['RI'], errors='coerce').replace(-9999, np.nan)
    tmp = tmp.dropna(subset=['_Dp', '_RI'])
    return [(float(dp), pd.Series(part['_RI'].to_numpy(), index=part.index).sort_index()) for dp, part in tmp.groupby('_Dp')]


def source_spans(df: pd.DataFrame, value: int) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    flagged = df[df['reference_source_flag'] == value]
    if flagged.empty:
        return []
    groups = (flagged.index.to_series().diff() > pd.Timedelta('90s')).cumsum()
    return [(part['t_start'].min(), part['t_end'].max()) for _, part in flagged.groupby(groups)]


def apply_time_axis(ax, *, show_xlabel: bool) -> None:
    locator = mdates.AutoDateLocator(minticks=3, maxticks=6)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    if show_xlabel:
        ax.set_xlabel('Time (UTC)')
    else:
        ax.tick_params(axis='x', labelbottom=False)


def numeric_column(df: pd.DataFrame, name: str) -> pd.Series:
    if name not in df:
        return pd.Series(dtype=float)
    return pd.to_numeric(df[name], errors='coerce').replace([-999.9, -9999], np.nan).dropna()


def read_reference_data_for_day(day: str) -> tuple[pd.DataFrame, pd.DataFrame, list[tuple[float, pd.Series]]]:
    dash_ict = DATA_DIR / 'DASH-SP' / f'ARCSIX-DASH_P3B_{day}_R1.ict'
    dlh_ict = DATA_DIR / 'DLH-H2O' / f'ARCSIX-DLH-H2O_P3B_{day}_R0.ict'
    if dash_ict.exists():
        dash = drop_timezone_from_index(read_ict_file(dash_ict, make_index=True))
        dash_groups = dash_groups_by_dp(dash)
    else:
        print(f'Skipping DASH overlay; file not found: {dash_ict}')
        dash = pd.DataFrame()
        dash_groups = []
    if dlh_ict.exists():
        dlh = drop_timezone_from_index(read_ict_file(dlh_ict, make_index=True))
    else:
        print(f'Skipping DLH RH overlay; file not found: {dlh_ict}')
        dlh = pd.DataFrame()
    return dash, dlh, dash_groups


def plot_retrieval_timeseries_for_day(day: str) -> Path:
    date = date_dash(day)
    nc_path = PRODUCT_ROOT / date / f'{date}_sizedist_merged.nc'
    out = PRODUCT_ROOT / f'{day}_RI_APSdensity_AmbientRH_3panel.png'
    product_df = merge_product_frame(nc_path)
    dash_df, dlh_df, dash_groups = read_reference_data_for_day(day)

    uhsas_raw = pd.to_numeric(product_df['uhsas_n'], errors='coerce').dropna()
    pops_raw = pd.to_numeric(product_df['pops_n'], errors='coerce').dropna()
    density_raw = pd.to_numeric(product_df['aps_density'], errors='coerce').dropna()
    rhw_raw = numeric_column(dlh_df, 'RHw_DLH')
    rhi_raw = numeric_column(dlh_df, 'RHi_DLH')

    uhsas_ma = moving_average(ri_for_average(uhsas_raw))
    pops_ma = moving_average(ri_for_average(pops_raw))
    density_ma = moving_average(density_raw)
    rhw_ma = moving_average(rhw_raw)
    rhi_ma = moving_average(rhi_raw)

    tmin = product_df.index.min()
    tmax = product_df.index.max()
    fims_count = int((product_df['reference_source_flag'] == 0).sum())
    putls_count = int((product_df['reference_source_flag'] == 1).sum())

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(9.0, 8.4), sharex=True, gridspec_kw={'hspace': 0.15})
    for ax in (ax1, ax2, ax3):
        for start, end in source_spans(product_df, 1):
            ax.axvspan(start, end, color='tab:blue', alpha=0.06, lw=0)

    ax1.scatter(uhsas_raw.index, uhsas_raw.values, color='red', s=5, alpha=0.18, linewidths=0)
    ax1.scatter(pops_raw.index, pops_raw.values, color='purple', s=5, alpha=0.18, linewidths=0)
    plot_segmented_line(ax1, uhsas_ma, color='red', lw=1.5, alpha=0.55, label=f'UHSAS MA {MA_WINDOW}')
    plot_segmented_line(ax1, pops_ma, color='purple', lw=1.5, alpha=0.55, label=f'POPS MA {MA_WINDOW}')

    greens = plt.get_cmap('Greens')(np.linspace(0.35, 0.90, max(len(dash_groups), 1)))
    dash_handles = []
    for (dp, series), color in zip(dash_groups, greens):
        ax1.plot(series.index, series.values, color=color, alpha=0.20, lw=1.0)
        ax1.scatter(series.index, series.values, color=color, s=10, alpha=0.95, linewidths=0)
        dash_handles.append(Line2D([0], [0], marker='o', linestyle='-', color=color, alpha=0.8, markersize=5, lw=1.0, label=f'DASH {int(round(dp))} nm'))

    ax1.set_ylabel('RI (real)')
    ax1.set_title(f'ARCSIX P-3B {day} - 1-min merge, FIMS or PUTLS-MERGE reference')
    ax1.grid(True, which='both', alpha=0.25)
    ax1.legend(
        handles=[
            Line2D([0], [0], marker='o', linestyle='None', color='red', markersize=5, label='UHSAS 1054 raw'),
            Line2D([0], [0], marker='o', linestyle='None', color='purple', markersize=5, label='POPS 405 raw'),
            Line2D([0], [0], linestyle='-', color='red', lw=1.2, alpha=0.8, label=f'UHSAS MA {MA_WINDOW}'),
            Line2D([0], [0], linestyle='-', color='purple', lw=1.2, alpha=0.8, label=f'POPS MA {MA_WINDOW}'),
        ] + dash_handles + [Patch(facecolor='tab:blue', alpha=0.10, label=f'PUTLS-MERGE ref ({putls_count}); FIMS ref ({fims_count})')],
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        frameon=True,
        fontsize=8,
    )

    ax2.scatter(density_raw.index, density_raw.values, color='black', s=5, alpha=0.18, linewidths=0, label='raw')
    plot_segmented_line(ax2, density_ma, color='black', lw=1.5, alpha=0.65, label=f'MA {MA_WINDOW}')
    ax2.set_ylabel(r'APS density (kg m$^{-3}$)')
    ax2.grid(True, which='both', alpha=0.25)
    ax2.legend(loc='best', fontsize=8)

    ax3.scatter(rhw_raw.index, rhw_raw.values, color='blue', s=5, alpha=0.18, linewidths=0)
    plot_segmented_line(ax3, rhw_ma, color='blue', lw=1.5, alpha=0.70, label=f'RHw MA {MA_WINDOW}')
    ax3.scatter(rhi_raw.index, rhi_raw.values, color='cyan', s=5, alpha=0.18, linewidths=0)
    plot_segmented_line(ax3, rhi_ma, color='cyan', lw=1.2, alpha=0.65, label=f'RHi MA {MA_WINDOW}')
    ax3.set_ylabel('Ambient RH (%)')
    ax3.set_ylim(0, 110)
    ax3.grid(True, which='both', alpha=0.25)
    ax3.legend(loc='best', fontsize=8)

    for ax in (ax1, ax2, ax3):
        ax.set_xlim(tmin, tmax)
    apply_time_axis(ax1, show_xlabel=False)
    apply_time_axis(ax2, show_xlabel=False)
    apply_time_axis(ax3, show_xlabel=True)

    fig.savefig(out, dpi=200, bbox_inches='tight', pad_inches=0.05)
    plt.close(fig)
    print(f'Saved: {out}')
    return out


RETRIEVAL_PLOT_DATES = RUN_DATES
retrieval_plot_paths = [plot_retrieval_timeseries_for_day(day) for day in RETRIEVAL_PLOT_DATES]
print(f'Wrote {len(retrieval_plot_paths)} retrieved-parameter plots.')

## 10. Campaign-Wide Instrument Comparison

This cell makes publication-style campaign comparison plots from the final accepted product. The merged spectra come from the final ICARTT files, not from raw daily merge output. The ICARTT rows are the master rows: individual instrument spectra are included only for those same merged periods. The code then matches each ICARTT row back to the QC NetCDF only to read `inlet_incloud_flag` and the already saved aligned/converted instrument spectra.

Rows are restricted to `reference_source_flag == 0` and `inlet_incloud_flag == 0`, so the plots compare FIMS-reference, inlet-clear merged periods only. For the original-bin plots, each instrument stays on its native/default diameter bins. For the converted-grid plots, the aligned arrays stored in the QC NetCDF are used on the product grid. The statistic table is built from accepted merged minutes first: row `k` is the same minute for the merged product and every instrument. No extra row mask is created from diameter-bin matching. For the converted-grid plots, the statistic uses the same per-bin comparison support for the merged product and converted instrument inputs: a row/bin is compared only when the merged ICARTT product exists there and at least one converted instrument has a positive value there. Values are not replaced or forced equal; the merged product still comes from the ICARTT columns and the instrument curves still come from the QC NetCDF aligned arrays. Geomean and median centers use positive values only because log-scale plotting cannot represent zero.


In [3]:

from sizedistmerge.ict_utils import read_aps, read_fims, read_ict_file, read_pops, read_uhsas
import arcsix_production.arcsix_merge_production as mp

START_NM = 20.0
END_NM = 6000.0
FIMS_REFERENCE_FLAG = 0
INLET_CLEAR_FLAG = 0
INSTRUMENTS = ('FIMS', 'UHSAS', 'POPS', 'APS')
PRODUCT_NAME = 'ARCSIX-MERGED-SIZEDIST-1min_P3B'

ict_dir = PRODUCT_ROOT / 'icartt_from_qc_flagged_nc' / PRODUCT_NAME
qc_nc_dir = PRODUCT_ROOT / 'qc_flagged_nc'
plot_dir = PRODUCT_ROOT / 'qc_plots'
plot_dir.mkdir(parents=True, exist_ok=True)

ict_paths = sorted(ict_dir.glob(f'{PRODUCT_NAME}_*_R1.ict'))
qc_nc_paths = sorted(qc_nc_dir.glob('*_sizedist_merged.nc'))
if not ict_paths:
    raise FileNotFoundError(f'No ICARTT product files found in {ict_dir}')
if not qc_nc_paths:
    raise FileNotFoundError(f'No QC NetCDF files found in {qc_nc_dir}')

qc_by_date = {path.name.split('_')[0].replace('-', ''): path for path in qc_nc_paths}
bin_log = plot_dir / 'campaign_validation_bin_counts_log.txt'
scratch_bin_log = Path('/private/tmp/campaign_validation_make_filtered_specs_log.txt')
bin_log.write_text('Campaign validation row-support log\n')
scratch_bin_log.write_text('')

native_rows = {name: [] for name in INSTRUMENTS}
native_mids = {}
converted_rows = {name: [] for name in ('Merged', *INSTRUMENTS)}
product_mids = None
period_count = 0


def seconds_of_day(times):
    times = pd.to_datetime(times)
    return (
        times.hour * 3600.0
        + times.minute * 60.0
        + times.second
        + times.microsecond / 1e6
    )


def centers_from_ict_header(path):
    for line in Path(path).read_text(encoding='utf-8', errors='ignore').splitlines():
        line = line.strip()
        if line.startswith('FINE_CENTERS_NM:'):
            return np.asarray([float(x.strip()) for x in line.split(':', 1)[1].split(',')], float)
    raise ValueError(f'{path.name}: missing FINE_CENTERS_NM header line')


def product_spectrum_from_ict(path):
    mids = centers_from_ict_header(path)
    df = read_ict_file(path, keep_time_col=True)
    cols = [f'DNLOG_{i:03d}' for i in range(1, len(mids) + 1)]
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise ValueError(f'{path.name}: missing merged spectrum columns {missing[:3]}')

    values = np.asarray(df[cols], float)
    values = np.where(values > -9000.0, values, np.nan)
    return df, mids, values


def load_qc_day(date):
    nc_path = qc_by_date.get(date)
    if nc_path is None:
        raise FileNotFoundError(f'No QC NetCDF file for {date}')

    ds = Dataset(nc_path)
    base_time = pd.Timestamp(ds.getncattr('base_time_iso'))
    starts = base_time + pd.to_timedelta(np.asarray(ds.variables['time_start_since_base_s'][:], float), unit='s')
    stops = base_time + pd.to_timedelta(np.asarray(ds.variables['time_end_since_base_s'][:], float), unit='s')
    start_s = seconds_of_day(starts)
    stop_s = seconds_of_day(stops)
    row_by_time = {
        (round(float(t0), 3), round(float(t1), 3)): i
        for i, (t0, t1) in enumerate(zip(start_s, stop_s))
    }
    return ds, starts, stops, row_by_time


def load_instruments(date):
    frames = {
        'FIMS': read_fims(DATA_DIR / 'FIMS' / f'ARCSIX-FIMS_P3B_{date}_R0.ict'),
        'UHSAS': read_uhsas(DATA_DIR / 'PUTLS-UHSAS' / f'ARCSIX-PUTLS-UHSAS_P3B_{date}_R1.ict'),
        'POPS': read_pops(DATA_DIR / 'PUTLS-POPS' / f'ARCSIX-PUTLS-POPS_P3B_{date}_R1.ict'),
        'APS': read_aps(DATA_DIR / 'LARGE-APS' / f'ARCSIX-LARGE-APS_P3B_{date}_R0.ict'),
    }
    for name, df in frames.items():
        if isinstance(df.index, pd.DatetimeIndex) and df.index.tz is not None:
            df = df.copy()
            df.index = df.index.tz_localize(None)
            frames[name] = df
    frames['FIMS'] = frames['FIMS'].copy()
    frames['FIMS'].index = frames['FIMS'].index - pd.Timedelta(seconds=FIMS_LAG)
    return frames


def add_native_spectrum(name, mids, values):
    mids = np.asarray(mids, float)
    values = np.asarray(values, float)
    if name not in native_mids:
        native_mids[name] = mids
    elif not np.allclose(native_mids[name], mids):
        raise ValueError(f'{name} native bins changed across the campaign')
    native_rows[name].append(values)


def finite_nonnegative(values):
    values = np.asarray(values, float)
    return np.where(np.isfinite(values) & (values >= 0.0), values, np.nan)


def finite_positive(values):
    values = np.asarray(values, float)
    return np.where(np.isfinite(values) & (values > 0.0), values, np.nan)


def geomean_stats(values, row_mask=None):
    # Rows are already aligned by accepted merged minute. The geometric mean is
    # computed from positive values because zero cannot be represented in log
    # space. Optional masks are only for invalid product-bin support, not for
    # changing the accepted-minute population.
    values = finite_positive(values)
    if row_mask is not None:
        values = np.where(row_mask, values, np.nan)
    n = np.sum(np.isfinite(values), axis=0)
    logs = np.log(values)

    center = np.full(values.shape[1], np.nan)
    log_sigma = np.full(values.shape[1], np.nan)
    use = n > 0
    center[use] = np.exp(np.nanmean(logs[:, use], axis=0))
    log_sigma[n == 1] = 0.0
    many = n > 1
    log_sigma[many] = np.nanstd(logs[:, many], axis=0)

    spread = np.exp(log_sigma)
    lo = center / spread
    hi = center * spread
    return center, lo, hi, n


def median_stats(values, row_mask=None):
    # Center: positive-value median. Band: linear 25th to 75th percentile
    # range from positive values. Zero-inclusive medians are not useful for
    # this log-y validation plot because they make sparse APS tail bins vanish.
    values = finite_positive(values)
    if row_mask is not None:
        values = np.where(row_mask, values, np.nan)
    n = np.sum(np.isfinite(values), axis=0)
    center = np.full(values.shape[1], np.nan)
    lo = np.full(values.shape[1], np.nan)
    hi = np.full(values.shape[1], np.nan)
    use = n > 0
    if np.any(use):
        center[use] = np.nanmedian(values[:, use], axis=0)
        quartiles = np.nanpercentile(values[:, use], [25, 75], axis=0)
        lo[use] = quartiles[0]
        hi[use] = quartiles[1]
    return center, lo, hi, n


for ict_path in ict_paths:
    date = ict_path.stem.split('_')[-2]
    frames = load_instruments(date)
    ict_df, ict_mids, ict_merged = product_spectrum_from_ict(ict_path)

    if product_mids is None:
        product_mids = ict_mids
    elif not np.allclose(product_mids, ict_mids, rtol=2e-4, atol=1e-3):
        raise ValueError(f'{ict_path.name}: product diameter grid changed')

    ds, qc_starts, qc_stops, row_by_time = load_qc_day(date)
    try:
        fine_edges = np.asarray(ds.variables['fine_edges_nm'][:], float)
        qc_mids = np.sqrt(fine_edges[:-1] * fine_edges[1:])
        if not np.allclose(qc_mids, ict_mids, rtol=2e-4, atol=1e-3):
            raise ValueError(f'{date}: ICT and QC NetCDF product grids do not match')

        ref_flags = np.asarray(ict_df['reference_source_flag'], int)
        inlet_flags = np.asarray(ds.variables['inlet_incloud_flag'][:], int)
        qc_rows = []
        ict_rows = []
        for ict_i, row in enumerate(ict_df.itertuples(index=False)):
            key = (round(float(row.Time_Start), 3), round(float(row.Time_Stop), 3))
            qc_i = row_by_time.get(key)
            if qc_i is None:
                raise RuntimeError(f'{date}: no QC NetCDF row for ICT time window {key}')
            if ref_flags[ict_i] == FIMS_REFERENCE_FLAG and inlet_flags[qc_i] == INLET_CLEAR_FLAG:
                ict_rows.append(ict_i)
                qc_rows.append(qc_i)

        if not ict_rows:
            continue

        ict_rows = np.asarray(ict_rows, int)
        qc_rows = np.asarray(qc_rows, int)
        converted_rows['Merged'].append(ict_merged[ict_rows])
        converted_rows['FIMS'].append(np.asarray(ds.variables['fims_aligned_dNdlogDp'][:], float)[qc_rows])
        converted_rows['UHSAS'].append(np.asarray(ds.variables['uhsas_aligned_dNdlogDp'][:], float)[qc_rows])
        converted_rows['POPS'].append(np.asarray(ds.variables['pops_aligned_dNdlogDp'][:], float)[qc_rows])
        converted_rows['APS'].append(np.asarray(ds.variables['aps_aligned_dNdlogDp'][:], float)[qc_rows])

        for qc_i in qc_rows:
            t_start = qc_starts[qc_i]
            t_stop = qc_stops[qc_i]
            chunk = {name: frames[name].loc[t_start:t_stop] for name in INSTRUMENTS}
            specs, _, _, _ = mp.make_filtered_specs(chunk, scratch_bin_log, order=INSTRUMENTS)
            missing = [name for name in INSTRUMENTS if name not in specs]
            if missing:
                raise RuntimeError(f'Missing production spectra for {date} {t_start} to {t_stop}: {missing}')
            for name in INSTRUMENTS:
                mids, _edges, values, _sigma = specs[name]
                add_native_spectrum(name, mids, values)
            period_count += 1
    finally:
        ds.close()

if period_count == 0:
    raise RuntimeError('No FIMS-reference, inlet-clear ICARTT product periods were found')

converted = {name: np.vstack(rows) for name, rows in converted_rows.items()}
native = {name: np.vstack(rows) for name, rows in native_rows.items()}
row_counts = {name: values.shape[0] for name, values in {**converted, **native}.items()}
bad_counts = {name: n for name, n in row_counts.items() if n != period_count}
if bad_counts:
    raise RuntimeError(f'Validation arrays are not row-aligned to accepted merged periods: {bad_counts}')


def build_original_stats(stat_func):
    stats = {'Merged': (product_mids, *stat_func(converted['Merged']))}
    for name in INSTRUMENTS:
        stats[name] = (native_mids[name], *stat_func(native[name]))
    return stats


def converted_comparison_support():
    # Use the same row/bin comparison support for every converted-grid curve.
    # A row/bin is compared only when the final merged product exists and at
    # least one converted instrument input exists there. This masks support,
    # not values, so nothing is forced to match.
    support = np.isfinite(converted['Merged']) & (converted['Merged'] >= 0.0)
    any_instrument = np.zeros_like(support, dtype=bool)
    for name in INSTRUMENTS:
        any_instrument |= np.isfinite(converted[name]) & (converted[name] > 0.0)
    return support & any_instrument


def build_converted_geomean_stats():
    support = converted_comparison_support()
    return {
        name: (product_mids, *geomean_stats(converted[name], support))
        for name in ('Merged', *INSTRUMENTS)
    }


def build_converted_median_stats():
    support = converted_comparison_support()
    return {
        name: (product_mids, *median_stats(converted[name], support))
        for name in ('Merged', *INSTRUMENTS)
    }


original_geomean_stats = build_original_stats(geomean_stats)
converted_geomean_stats = build_converted_geomean_stats()
original_median_stats = build_original_stats(median_stats)
converted_median_stats = build_converted_median_stats()

styles = {
    'Merged': {'color': 'black', 'lw': 4.0, 'ls': '-', 'alpha': 1.0},
    'FIMS': {'color': 'tab:red', 'lw': 2.5, 'ls': '--', 'alpha': 0.95},
    'UHSAS': {'color': 'tab:green', 'lw': 2.5, 'ls': '--', 'alpha': 0.95},
    'POPS': {'color': 'tab:purple', 'lw': 2.5, 'ls': '--', 'alpha': 0.95},
    'APS': {'color': 'tab:blue', 'lw': 2.5, 'ls': '--', 'alpha': 0.95},
}


def draw_campaign_comparison(stats, out_stem):
    fig, ax = plt.subplots(figsize=(4.6, 4.6), dpi=300, constrained_layout=True)
    for name in ('Merged', *INSTRUMENTS):
        mids, center, lo, hi, n = stats[name]
        use = (
            (mids >= START_NM)
            & (mids <= END_NM)
            & np.isfinite(center)
            & np.isfinite(lo)
            & np.isfinite(hi)
            & (center > 0.0)
            & (lo > 0.0)
            & (hi > 0.0)
        )
        fill_zorder = 1 if name == 'Merged' else 3
        line_zorder = 2 if name == 'Merged' else 4
        ax.fill_between(
            mids[use], lo[use], hi[use],
            color=styles[name]['color'], alpha=0.18, linewidth=0, zorder=fill_zorder,
        )
        ax.plot(mids[use], center[use], label=name, zorder=line_zorder, **styles[name])

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(START_NM, END_NM)
    ax.set_box_aspect(1)
    ax.set_xlabel('Diameter (nm)')
    ax.set_ylabel(r'dN/dlog$D_p$ (# cm$^{-3}$)')
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(frameon=True, ncol=1, loc='upper right', fontsize=10)

    png = plot_dir / f'{out_stem}.png'
    pdf = plot_dir / f'{out_stem}.pdf'
    fig.savefig(png, dpi=300)
    fig.savefig(pdf)
    plt.close(fig)
    return png, pdf

original_geomean_plot = draw_campaign_comparison(
    original_geomean_stats,
    'campaign_merged_period_original_instrument_comparison_geomean',
)
converted_geomean_plot = draw_campaign_comparison(
    converted_geomean_stats,
    'campaign_merged_period_converted_instrument_comparison_geomean',
)
original_median_plot = draw_campaign_comparison(
    original_median_stats,
    'campaign_merged_period_original_instrument_comparison_median',
)
converted_median_plot = draw_campaign_comparison(
    converted_median_stats,
    'campaign_merged_period_converted_instrument_comparison_median',
)

with bin_log.open('a') as f:
    f.write(f'ICARTT files: {len(ict_paths)}\n')
    f.write(f'FIMS-reference, inlet-clear merged periods: {period_count}\n')
    f.write('All arrays below start from the same accepted merged-minute rows; per-bin counts only drop invalid/nonpositive values for that statistic.\n')
    for name in ('Merged', *INSTRUMENTS):
        n = original_geomean_stats[name][4]
        f.write(f'original {name}: min rows/bin={int(np.nanmin(n))}, max rows/bin={int(np.nanmax(n))}\n')
        n = converted_geomean_stats[name][4]
        f.write(f'converted {name}: min rows/bin={int(np.nanmin(n))}, max rows/bin={int(np.nanmax(n))}\n')

print(f'ICARTT files: {len(ict_paths)}')
print(f'FIMS-reference, inlet-clear merged periods: {period_count}')
print('Original-bin geomean plot:', original_geomean_plot[0])
print('Converted-grid geomean plot:', converted_geomean_plot[0])
print('Original-bin median plot:', original_median_plot[0])
print('Converted-grid median plot:', converted_median_plot[0])


QC NetCDF files: 19
FIMS-reference, inlet-clear merged periods: 4475
Merged positive rows by bin: min=0, max=4475
FIMS positive rows by bin: min=991, max=4475
UHSAS positive rows by bin: min=0, max=4475
POPS positive rows by bin: min=937, max=4475
APS positive rows by bin: min=898, max=4460
Original-bin plot: /Users/C832577250/Output/arcsix_sizedist_merge_full_1min_fims_or_putls_temporal_ri0p1_rho5em7/qc_plots/campaign_merged_period_original_instrument_comparison_geomean.png
Converted-grid plot: /Users/C832577250/Output/arcsix_sizedist_merge_full_1min_fims_or_putls_temporal_ri0p1_rho5em7/qc_plots/campaign_merged_period_converted_instrument_comparison_geomean.png


## 11. Production Summary

Use this cell after running the workflow to confirm where the important artifacts were written.

In [ ]:
print('Product root:', PRODUCT_ROOT)
print('Daily NetCDF files:', len(mp.find_merged_netcdf_files(PRODUCT_ROOT)))
print('QC NetCDF files:', len(sorted((PRODUCT_ROOT / 'qc_flagged_nc').glob('*_sizedist_merged.nc'))))
print('ICARTT files:', len(sorted((PRODUCT_ROOT / 'icartt_from_qc_flagged_nc').glob('**/*.ict'))))
print('QC plots:', PRODUCT_ROOT / 'qc_plots')
print('ICARTT inspection plots:', INSPECTION_PLOT_DIR)